## Dadi Lab

In [79]:
two = 2
three = 3
two*three

6

In [80]:
import msprime
import dadi
import numpy as np

In [81]:
demography = msprime.Demography()

In [82]:
demography.add_population(name="pop", initial_size=500)  # present size

# ancestral size before T=800 generations ago was 10,000
demography.add_population_parameters_change(
    time=800, initial_size=10_000, population="pop"
)

PopulationParametersChange(time=800, initial_size=10000, growth_rate=None, population='pop')

In [83]:
demography

Demography(populations=[Population(initial_size=500, growth_rate=0, name='pop', description='', extra_metadata={}, default_sampling_time=None, initially_active=None, id=0)], events=[PopulationParametersChange(time=800, initial_size=10000, growth_rate=None, population='pop')], migration_matrix=array([[0.]]))

In [84]:
ts = msprime.sim_ancestry(
    samples={"pop": 20},
    demography=demography,
    sequence_length=5_000_000,
    recombination_rate=1e-8,
    random_seed=42,
)

In [85]:
mts = msprime.sim_mutations(ts, rate=1e-8, random_seed=43)

In [86]:
mts.genotype_matrix()
with open("bottleneck_sim.vcf", "w") as f:
    mts.write_vcf(f)

In [87]:
vcf_file = "bottleneck_sim.vcf"
popfile = "pops.txt"
dd = dadi.Misc.make_data_dict_vcf(vcf_file, "pops.txt")
#print(dd.keys())

In [90]:
#fs = dadi.Spectrum.from_data_dict(dd, ['pop'], projections = [30], polarized = False)
fs = dadi.Spectrum.from_data_dict(
    dd,
    pop_ids=["pop"],
    projections=[30],
    polarized=False,   # folded SFS
)
print("Spectrum sample size:", fs.sample_sizes)
print("Segregating sites:", fs.S())

KeyError: 'pop'

In [50]:
def snm(params, ns, pts): # define single population model, no free parameters
    xx = dadi.Numerics.default_grid(pts)
    phi = dadi.PhiManip.phi_1D(xx)
    fs_model = dadi.Spectrum.from_phi(phi, ns, (xx,))
    return fs_model

def two_epoch(params, ns, pts): # define bottlenneck model
    nu, T = params # two free parameters: scaled current pop size and time of split 
    xx = dadi.Numerics.default_grid(pts)
    phi = dadi.PhiManip.phi_1D(xx)
    phi = dadi.Integration.one_pop(phi, xx, T, nu)
    fs_model = dadi.Spectrum.from_phi(phi, ns, (xx,))
    return fs_model

In [51]:
pts_l = [40, 50, 60]
snm_ex = dadi.Numerics.make_extrap_log_func(snm)
two_epoch_ex = dadi.Numerics.make_extrap_log_func(two_epoch)

In [52]:
model_snm = snm_ex([], fs.sample_sizes, pts_l)
theta_snm = dadi.Inference.optimal_sfs_scaling(model_snm, fs)
ll_snm = dadi.Inference.ll_multinom(model_snm, fs)

print("\nConstant-size model")
print("log-likelihood:", ll_snm)
print("theta:", theta_snm)

NameError: name 'fs' is not defined

In [ ]:
popt = dadi.Inference.optimize_log(
    p0_perturbed,
    fs,
    two_epoch_ex,
    pts_l,
    lower_bound=lower_bound,
    upper_bound=upper_bound,
    verbose=1,
    maxiter=50,
)

In [ ]:
model_two = two_epoch_ex(popt, fs.sample_sizes, pts_l)
theta_two = dadi.Inference.optimal_sfs_scaling(model_two, fs)
ll_two = dadi.Inference.ll_multinom(model_two, fs)

print("\nTwo-epoch model")
print("best params [nu, T]:", popt)
print("log-likelihood:", ll_two)
print("theta:", theta_two)

In [ ]:
print("\nModel comparison")
print(f"Delta log-likelihood (two-epoch - constant): {ll_two - ll_snm:.3f}")

# constant model has k=0 free params in this formulation
# two_epoch has k=2
aic_snm = 2 * 0 - 2 * ll_snm
aic_two = 2 * 2 - 2 * ll_two

print(f"AIC constant: {aic_snm:.3f}")
print(f"AIC two-epoch: {aic_two:.3f}")

In [ ]:
print("True parameters:")
print("nu =", 0.05)
print("T  =", 0.04)

print("\nInferred parameters:")
print("nu =", popt[0])
print("T  =", popt[1])

In [ ]:
N_anc = 10_000  # known from simulation
nu_est, T_est = popt # label popt parameter estimates
N_curr_est = nu_est * N_anc # multiply Nu in dadi units by ancestral populaiton size to get current N_e
t_est = T_est * 2 * N_anc # multiply T in dadi units by 2N_e to get generations
print("Estimated current size:", N_curr_est)
print("Estimated bottleneck time (generations):", t_est)

Unfortunately, I have been genuinely unable to get this lab to run properly. I have spent a long time trying to troubleshoot it and can't figure out what the fix is. I understand that it stems from the population labelling, but editing the file and the naming of variables in many different ways has nit changed the outcome.